# RNA+ATAC mapping and prediction

This tutorial trains CONNECT on paired RNA+ATAC data, then stores latent embeddings, mapped embeddings, and cross-modal predictions in AnnData objects.


## 1. Load data

Replace `data_dir` with the directory containing paired train/test `.h5ad` files.


In [ ]:
import os
from os.path import join

import scanpy as sc

from connect import (
    build_model,
    build_paired_loader,
    get_device,
    init_logger,
    make_modality,
    predict,
    save_model,
    save_outputs,
    set_seed,
    train_model,
)


In [ ]:
data_dir = "/path/to/rna_atac_dataset"
save_dir = "./connect_rna_atac_result"

set_seed(42)
device = get_device("0")
logger = init_logger(save_dir)

rna_train = sc.read_h5ad(join(data_dir, "rna_train.h5ad"))
atac_train = sc.read_h5ad(join(data_dir, "atac_train.h5ad"))
rna_test = sc.read_h5ad(join(data_dir, "rna_test.h5ad"))
atac_test = sc.read_h5ad(join(data_dir, "atac_test.h5ad"))


## 2. Describe preprocessing

RNA is used as provided here. ATAC uses the Andrew TF-IDF transform and optional count truncation.


In [ ]:
train_rna = make_modality(rna_train, "RNA", preprocess=False)
train_atac = make_modality(atac_train, "ATAC", preprocess="Andrew", ceiling=True)
test_rna = make_modality(rna_test, "RNA", preprocess=False)
test_atac = make_modality(atac_test, "ATAC", preprocess="Andrew", ceiling=True)


## 3. Build loaders and model


In [ ]:
data_loader, valid_data_loader = build_paired_loader(
    train_rna,
    train_atac,
    batch_size=128,
    validation_split=0.1,
    num_workers=2,
    training=True,
    logger=logger,
)

model = build_model(data_loader, train_rna, train_atac, device=device)
logger.info(model)


## 4. Standard paired training


In [ ]:
model = train_model(
    model,
    data_loader,
    valid_data_loader,
    device=device,
    epochs=80,
    lr=1e-3,
    weight_decay=1e-3,
    amsgrad=True,
    logger=logger,
)


## 5. Infer mapping and prediction outputs


In [ ]:
outputs = predict(
    model,
    test_rna,
    test_atac,
    device=device,
    batch_size=128,
    num_workers=2,
    logger=logger,
)

rna_test.obsm["connect_rna_emb"] = outputs["mod1_latent"].cpu().numpy()
rna_test.obsm["connect_rna_mapped_from_atac"] = outputs["mod1_mapping_from_mod2"].cpu().numpy()
rna_test.layers["connect_rna_predicted_from_atac"] = outputs["mod1_predicted_from_mod2"].cpu().numpy()

atac_test.obsm["connect_atac_emb"] = outputs["mod2_latent"].cpu().numpy()
atac_test.obsm["connect_atac_mapped_from_rna"] = outputs["mod2_mapping_from_mod1"].cpu().numpy()
atac_test.layers["connect_atac_predicted_from_rna"] = outputs["mod2_predicted_from_mod1"].cpu().numpy()

save_model(model, save_dir)
save_outputs(outputs, save_dir, filename="rna_atac_outputs.pt")
